# multiprocessing.Value 和 Array

`Value` 用于共享单个标量，`Array` 用于共享固定长度数组。复合操作不是原子的，即使启用了自动锁，也应该在读改写时显式获取锁。

| 工具 | 适合场景 |
| --- | --- |
| `Value` | 跨进程共享单个整数、浮点数等标量 |
| `RawValue` | 不带自动同步的底层标量共享 |
| `Array` | 跨进程共享固定长度数组 |
| `RawArray` | 不带自动同步的底层数组共享 |


In [ ]:
import multiprocessing
import time


def increment_value(shared_value) -> None:
    for _ in range(5):
        time.sleep(0.2)
        with shared_value.get_lock():
            shared_value.value += 1
            print(f'Shared value in process: {shared_value.value}')


def increment_array(shared_array, index: int) -> None:
    for _ in range(5):
        time.sleep(0.2)
        with shared_array.get_lock():
            shared_array[index] += 1
            print(f'Shared array[{index}] in process: {shared_array[index]}')


if __name__ == '__main__':
    shared_value = multiprocessing.Value('i', 0)
    value_processes = [multiprocessing.Process(target=increment_value, args=(shared_value,)) for _ in range(3)]
    for process in value_processes:
        process.start()
    for process in value_processes:
        process.join()
    print(f'Final shared value: {shared_value.value}')

    shared_array = multiprocessing.Array('i', [0, 0, 0], lock=True)
    array_processes = [multiprocessing.Process(target=increment_array, args=(shared_array, index)) for index in range(3)]
    for process in array_processes:
        process.start()
    for process in array_processes:
        process.join()
    print(f'Final shared array: {shared_array[:]}')
